# Client maker
<br>This scipt makes the csv of the clients with their preferences and address. 
<br>The addresses need to be calculated first in the data_generation/address_generator.ipynb file.

In [30]:
import random
import csv
from pathlib import Path
import osmnx as ox
import pandas as pd

In [31]:
heerlen_addresses_df = pd.read_csv("../output/heerlen_addresses.csv")

In [32]:
# Use prevalidated addresses from heerlen_addresses_df.
required_columns = {"full_address", "coordinates"}
missing_columns = required_columns - set(heerlen_addresses_df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns in heerlen_addresses_df: {sorted(missing_columns)}")

VALID_HEERLEN_ADDRESS_ROWS = (
    heerlen_addresses_df[["full_address", "coordinates"]]
    .dropna(subset=["full_address"])
    .assign(full_address=lambda d: d["full_address"].astype(str).str.strip())
    .drop_duplicates(subset=["full_address"])
    .to_dict("records")
)

if not VALID_HEERLEN_ADDRESS_ROWS:
    raise ValueError("No usable rows found in heerlen_addresses_df['full_address'].")

# Shuffle rows and cycle through them to avoid duplicate addresses until all are used.
random.shuffle(VALID_HEERLEN_ADDRESS_ROWS)
_ADDRESS_INDEX = 0

def generate_real_address(max_attempts: int = 1):
    """Return one known Heerlen address row with its coordinates."""
    global _ADDRESS_INDEX
    row = VALID_HEERLEN_ADDRESS_ROWS[_ADDRESS_INDEX % len(VALID_HEERLEN_ADDRESS_ROWS)]
    _ADDRESS_INDEX += 1
    return row

# Possible care arrangements
CARE_ARRANGEMENTS = ["HBH Basic", "HBH Plus", "Wash & Ironing", "V&V"]
CARE_HOURS = [1, 1.5, 2, 2.5, 3]   # step 0.5

def generate_client(index):
    """Generate a single client record as a dictionary."""
    name = f"Client {index}"
    address_row = generate_real_address()
    address = address_row["full_address"]
    coordinates = address_row.get("coordinates")
    care_arrangement = random.choice(CARE_ARRANGEMENTS)
    preference = random.choice(["morning", "afternoon"])

    if preference == "morning":
        time_window_start = "08:00"
        time_window_end   = "12:00"
    else:
        time_window_start = "12:00"
        time_window_end   = "18:00"

    care_hours = random.choice(CARE_HOURS)

    # Most clients have no pets; occasional 1 or 2
    dogs = random.choices([0, 1, 2], weights=[0.7, 0.2, 0.1])[0]
    cats = random.choices([0, 1, 2], weights=[0.6, 0.3, 0.1])[0]

    # 30% chance of smoking
    smokes = random.random() < 0.3

    return {
        "name": name,
        "address": address,
        "coordinates": coordinates,
        "care_arrangement": care_arrangement,
        "preferences": preference,
        "time_window_start": time_window_start,
        "time_window_end": time_window_end,
        "care_hours": care_hours,
        "dogs": dogs,
        "cats": cats,
        "smokes": smokes,
    }

def generate_clients_csv(num_clients, filename="../output/clients.csv"):
    """Generate num_clients records and write them to a CSV file."""
    fieldnames = [
        "name", "address", "coordinates", "care_arrangement", "preferences",
        "time_window_start", "time_window_end", "care_hours",
        "dogs", "cats", "smokes"
    ]

    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, mode="w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for i in range(1, num_clients + 1):
            client = generate_client(i)
            # Convert boolean to lowercase string for readability
            client["smokes"] = str(client["smokes"]).lower()
            writer.writerow(client)

    print(f"Generated {num_clients} clients in '{output_path}'.")

if __name__ == "__main__":
    # Generate 100 clients by default
    generate_clients_csv(100)

Generated 100 clients in '..\output\clients.csv'.
